# Used Volkswagen Market Analysis — Latvia

## Project overview

This case study explores a historical snapshot of public Volkswagen vehicle advertisements from a Latvian classified-advertisement website. The original Python exercise used `pandas.read_html()` to collect public listing tables. For the portfolio version, a static snapshot is included so that the analysis remains reproducible even if the website structure changes.

### Analytical questions

- What does the price distribution look like in the captured listings?
- Which models appear most often in the snapshot?
- How strongly is listing price associated with vehicle year?
- How do prices differ between diesel and non-diesel listings?

> **Scope note:** this is a small historical snapshot, not a representative valuation model of the Latvian car market.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = Path("../data/volkswagen_listings_snapshot.csv")
df = pd.read_csv(DATA_PATH)

df.head()

## Data preparation

In [ ]:
df.info()

In [ ]:
# Derive a simple fuel-type indicator from the original engine notation.
df["fuel_type"] = np.where(
    df["engine"].astype(str).str.upper().str.endswith("D"),
    "Diesel",
    "Petrol / other"
)

df["price_eur"] = pd.to_numeric(df["price_eur"], errors="coerce")
df["year"] = pd.to_numeric(df["year"], errors="coerce")

df.isna().sum()

## Descriptive statistics

In [ ]:
price_stats = df["price_eur"].describe().round(2)
price_stats

In [ ]:
summary = pd.Series({
    "Listings": len(df),
    "Mean price (EUR)": df["price_eur"].mean(),
    "Median price (EUR)": df["price_eur"].median(),
    "Minimum price (EUR)": df["price_eur"].min(),
    "Maximum price (EUR)": df["price_eur"].max(),
}).round(2)

summary

The snapshot contains a wide spread of asking prices. Because several newer commercial and SUV models sit well above older vehicles, the mean is higher than the median. This is a useful reminder that the median is often a more robust summary for small, skewed listing samples.

## Price distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(df["price_eur"], bins=10, edgecolor="black")
ax.axvline(df["price_eur"].median(), linestyle="--", linewidth=1.5, label="Median")
ax.set_title("Distribution of Volkswagen asking prices")
ax.set_xlabel("Price (EUR)")
ax.set_ylabel("Number of listings")
ax.legend()
plt.tight_layout()
plt.show()

## Model frequency

In [ ]:
model_counts = df["model"].value_counts().head(10)

fig, ax = plt.subplots(figsize=(8, 4.5))
model_counts.sort_values().plot(kind="barh", ax=ax)
ax.set_title("Most frequent models in the snapshot")
ax.set_xlabel("Listings")
ax.set_ylabel("Model")
plt.tight_layout()
plt.show()

model_counts

## Vehicle year and asking price

In [ ]:
year_price_corr = df[["year", "price_eur"]].corr().loc["year", "price_eur"]
print(f"Pearson correlation between vehicle year and asking price: {year_price_corr:.3f}")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(df["year"], df["price_eur"], alpha=0.8)
ax.set_title("Vehicle year vs. asking price")
ax.set_xlabel("Vehicle year")
ax.set_ylabel("Price (EUR)")
plt.tight_layout()
plt.show()

The snapshot shows a positive association between vehicle year and asking price. This does **not** imply that year alone determines value: model, mileage, condition, specification and seller characteristics are not controlled for in this small dataset.

## Diesel vs. non-diesel listings

In [ ]:
fuel_summary = (
    df.groupby("fuel_type")["price_eur"]
      .agg(listings="size", mean_price="mean", median_price="median", min_price="min", max_price="max")
      .round(2)
)
fuel_summary

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
df.boxplot(column="price_eur", by="fuel_type", ax=ax)
ax.set_title("Price by fuel-type indicator")
ax.set_xlabel("")
ax.set_ylabel("Price (EUR)")
plt.suptitle("")
plt.tight_layout()
plt.show()

## Key findings

- The original 30-listing snapshot has a **median asking price of EUR 4,950** and a mean of approximately **EUR 8,388**.
- Asking prices range from **EUR 950 to EUR 28,800**, indicating a strongly heterogeneous sample.
- Newer vehicle years generally correspond to higher asking prices in this snapshot.
- Model mix and fuel type contribute to visible differences in price, but the sample is too small for market-wide inference.

## Limitations

The data represent a limited historical snapshot of public advertisements. The analysis does not control for mileage, vehicle condition, trim, transmission, seller type or duplicate/reposted advertisements. The results should therefore be interpreted as a demonstration of web-data cleaning and exploratory analysis rather than a valuation model.

## Optional: how the original data was collected

The original exercise used `pandas.read_html()` to read tables from public listing pages. A live scraper is intentionally not executed in this portfolio notebook because website HTML structures can change and automated access should respect the website's current terms and technical restrictions.

```python
# Illustrative pattern only:
# tables = pd.read_html("https://www.ss.com/en/transport/cars/volkswagen/")
# listings = tables[...]
```